In [2]:
!pip install stable-baselines3
!git clone https://github.com/aquacropos/aquacrop-gym.git
%cd aquacrop-gym


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.8 MB/s eta 0:00:00
Cloning into 'aquacrop-gym'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 129 (delta 26), reused 117 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 18.46 MiB | 21.06 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/aquacrop-gym


In [3]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import numpy as np
import random
import aquacropgym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
import aquacrop
import aquacropgym

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather, get_filepath
from aquacropgym.envs import CropEnv
weather_file_path = get_filepath('tunis_climate.txt')


ModuleNotFoundError: No module named 'aquacrop.classes'

In [56]:
import numpy as np
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Box

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather, get_filepath


class AquaCropEnv(Env):
    def __init__(self):
        super().__init__()

        self.action_space = Box(
            low=np.array([0], dtype=np.float32),
            high=np.array([999], dtype=np.float32),
            dtype=np.float32
        )

        # [soil_moisture, rain, temp, canopy_cover, biomass]
        self.observation_space = Box(
            low=np.array([0, 0, -20, 0, 0], dtype=np.float32),
            high=np.array([100, 50, 60, 100, 100000], dtype=np.float32),
            dtype=np.float32
        )

        self.season_length = 30
        self.day = 0
        self.total_water = 0
        self.state = None

        self.weather_file_path = get_filepath("tunis_climate.txt")
        self.weather_df = prepare_weather(self.weather_file_path)

    def _make_model(self):
        model = AquaCropModel(
            sim_start_time="1979/10/01",
            sim_end_time="1980/05/30",
            weather_df=self.weather_df,
            soil=Soil(soil_type="SandyLoam"),
            crop=Crop("Wheat", planting_date="10/01"),
            initial_water_content=InitialWaterContent(value=["FC"])
        )
        return model

    def reset(self, seed=None, options=None):
      super().reset(seed=seed)

      self.day = 0
      self.total_water = 0
      self.model = self._make_model()

      self.model.run_model(num_steps=1, initialize_model=True)

      self.state = self._get_state_from_aquacrop()

      return self.state, {}
    def step(self, action):
      irrigation = float(action[0])
      self.total_water += irrigation

      self.day += 1

      self.model.run_model(num_steps=1, initialize_model=False)

      self.state = self._get_state_from_aquacrop()

      soil_moisture, rain, temp, canopy_cover, biomass = self.state

      reward = biomass * 0.01 - irrigation * 0.05

      terminated = self.day >= self.season_length
      truncated = False

      info = {
          "day": self.day,
          "irrigation": irrigation,
          "total_water": self.total_water,
          "soil_moisture": soil_moisture,
          "biomass": biomass
      }

      return self.state, reward, terminated, truncated, info

    def _get_state_from_aquacrop(self):
      idx = self.day

      water_flux = self.model._outputs.water_flux
      water_storage = self.model._outputs.water_storage
      crop_growth = self.model._outputs.crop_growth

      last_flux = water_flux[idx]
      last_storage = water_storage[idx]
      last_crop = crop_growth[idx]

      print("DAY:", idx)
      print("storage:", last_storage)
      print("crop:", last_crop)

      # пример: последние значения storage похожи на soil moisture theta
      soil_moisture = float(last_storage[-1]) * 100

      # пока временно
      rain = float(last_flux[2]) if len(last_flux) > 2 else 0
      temp = 25

      # пока нужно подобрать индексы
      canopy_cover = float(last_crop[-3]) if len(last_crop) > 3 else 0
      biomass = float(last_crop[-2]) if len(last_crop) > 2 else 0

      state = np.array(
          [soil_moisture, rain, temp, canopy_cover, biomass],
          dtype=np.float32
      )

      return state

    def render(self):
        print(
            f"Day: {self.day}, "
            f"Soil moisture: {self.state[0]:.2f}, "
            f"Rain: {self.state[1]:.2f}, "
            f"Temp: {self.state[2]:.2f}, "
            f"Canopy: {self.state[3]:.2f}, "
            f"Biomass: {self.state[4]:.2f}"
        )

In [58]:
from gymnasium.utils.env_checker import check_env

env = AquaCropEnv()
check_env(env)

/usr/local/lib/python3.12/dist-packages/gymnasium/utils/env_checker.py:339: UserWarning: WARN: For Box action spaces, we recommend using a symmetric and normalized space (range=[-1, 1] or [0, 1]). See https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html for more information.
  logger.warn(


DAY: 0
storage: [0.         1.         1.         0.19077646 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 0.   0.   1.  21.5 21.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 0
storage: [0.         1.         1.         0.19077646 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 0.   0.   1.  21.5 21.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 0
storage: [0.         1.         1.         0.19077646 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 0.   0.   1.  21.5 21.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 0
storage: [0.         1.         1.         0.19077646 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 0.   0.   1.  21.5 21.5  0.3  0.   0.   0.   0.   0

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/gymnasium/utils/env_checker.py:440: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datet

In [40]:
model = AquaCropModel(
            sim_start_time=f"{1979}/10/01",
            sim_end_time=f"{1985}/05/30",
            weather_df=prepare_weather(weather_file_path),
            soil=Soil(soil_type='SandyLoam'),
            crop=Crop('Wheat', planting_date='10/01'),
            initial_water_content=InitialWaterContent(value=['FC']),
        )
# 1 день симуляции
model.run_model(num_steps=1)

# влажность по soil compartments / слоям
theta_layers = model._init_cond.th

# первый слой
theta_first_layer = theta_layers[0]

print(theta_first_layer)

0.19077646219099564


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [59]:
env = AquaCropEnv()

model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=5000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DAY: 0
storage: [0.         1.         1.         0.19077646 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 0.   0.   1.  21.5 21.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 1
storage: [1.         1.         2.         0.17307285 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 1.   0.   2.  20.  41.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 2
storage: [2.         1.         3.         0.15977257 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 2.   0.   3.  20.  61.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 3
storage: [3.         1.         4.         0.14649658 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 3.   0.   4.  20.  81.5  0.3  0.   0.   0.   0.   0

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DAY: 19
storage: [19.          1.         20.          0.05        0.18222261  0.22
  0.22        0.22        0.22        0.22        0.22        0.22
  0.22        0.22        0.22      ]
crop: [1.90000000e+01 0.00000000e+00 2.00000000e+01 1.75000000e+01
 4.18500000e+02 5.91647333e-01 9.51255500e-02 9.51255500e-02
 1.59900405e+01 1.60712931e+01 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00]
DAY: 20
storage: [20.         1.        21.         0.05       0.1764456  0.22
  0.22       0.22       0.22       0.22       0.22       0.22
  0.22       0.22       0.22     ]
crop: [2.00000000e+01 0.00000000e+00 2.10000000e+01 1.75000000e+01
 4.36000000e+02 6.09611258e-01 9.99037875e-02 9.99037875e-02
 1.84402673e+01 1.85215198e+01 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00]
DAY: 21
storage: [21.         1.        22.         0.05       0.1700474  0.22
  0.22       0.22       0.22       0.22       0.22       0.22
  0.22       0.22     

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DAY: 7
storage: [7.         1.         8.         0.10540411 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [  7.           0.           8.          20.         166.
   0.31429501   0.           0.           0.           0.
   0.           0.           0.           0.           0.        ]
DAY: 8
storage: [8.         1.         9.         0.09426027 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [  8.           0.           9.          21.5        187.5
   0.34666513   0.           0.           0.           0.
   0.           0.           0.           0.           0.        ]
DAY: 9
storage: [ 9.          1.         10.          0.08489359  0.22        0.22
  0.22        0.22        0.22        0.22        0.22        0.22
  0.22        0.22        0.22      ]
crop: [  9.           0.          10.          23.5        211.
   0.37555801   0.     

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DAY: 1
storage: [1.         1.         2.         0.17307285 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 1.   0.   2.  20.  41.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 2
storage: [2.         1.         3.         0.15977257 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 2.   0.   3.  20.  61.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 3
storage: [3.         1.         4.         0.14649658 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [ 3.   0.   4.  20.  81.5  0.3  0.   0.   0.   0.   0.   0.   0.   0.
  0. ]
DAY: 4
storage: [4.         1.         5.         0.13563303 0.22       0.22
 0.22       0.22       0.22       0.22       0.22       0.22
 0.22       0.22       0.22      ]
crop: [  4.    0.    5.   22.5 104.    0.3   0.    0.    0.

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
!pip uninstall -y aquacrop
!pip install aquacrop==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 77.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/d